In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr

sys.path.append('../')
from utils.visualization_utils import pathway_swarm_plot, get_data_pathways, get_data_ridge, plot_ridge_pathways

# Get the data

In [ ]:
important_pathways = {
    0: [18, 45, 29, 12, 30],
    1: [18, 45, 29, 12, 13],
    2: [12, 29, 45, 13, 17],
    3: [29, 45, 17, 18, 12],
    4: [29, 45, 18, 12, 30]
}

In [ ]:
## Fill in your own settings here!!

# Data type
type = 'brca'
nr = 512
nr_proto = 16

# Fold
fold = 2

# Name of experiment/model
exp_name = "DIMAFx"

In [ ]:
# Loading pathways names
hallmarks_pd = pd.read_csv('../data/data_files/hallmarks_signatures.csv')
hallmarks = np.array([' '.join(x[9:].split('_')) for x in hallmarks_pd.columns])
hallmarks_ids = np.array([f'R{i}' for i, item in enumerate (hallmarks)])
hallmarks_names = np.array([f'R{i}: {item}' for i, item in enumerate (hallmarks)])

rna_data = pd.read_csv(f'../data/data_files/tcga_{type}/rna/rna_data.csv', index_col=0)

# Obtain shap values
shap_results_fold_dir = f'../results/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/shap/modal/shap_all_test_512.pkl'
shap_dict = pickle.load(open(shap_results_fold_dir, 'rb')) 
shap_values = shap_dict['shap values']
shap_values_rna = np.sum(shap_values[:, nr_proto:, :], axis=2)

# Obtain test data
test_data = pd.read_csv(f'../data/data_files/tcga_{type}/splits/{fold}/test_filtered.csv')


In [ ]:
# Plot pathways
pathway_nums = important_pathways[fold]
print(pathway_nums)

plot_data = get_data_pathways(pathway_nums, test_data, hallmarks_pd, rna_data, shap_dict)
df = pathway_swarm_plot(plot_data)

In [ ]:
df

# Get correlation pathway expression & multimodal feature 

In [ ]:
# Obtain shap values
mult_shap_results_fold_dir = f'../results/ablations/dss_survival_{type}/{exp_name}/Fold_{fold}/post_training/shap/post_attn/shap_all_test_{nr}.pkl'
mult_shap_dict = pickle.load(open(mult_shap_results_fold_dir, 'rb')) 
mult_shap_values = np.sum(mult_shap_dict['shap values'], axis=2)
feats_names = mult_shap_dict['Feature names']

In [ ]:
# SHared W8 feature in fold 2 is at index 108
# SHared R48 feature in fold 2 is at index 98
print(feats_names[108])
shap_spec_feat = mult_shap_values[:, 108]
pathway = 13

In [ ]:
rna_vals = []
shap_vals = []
for i, (sample, slide) in enumerate(zip(test_data['case_id'].values, test_data['slide_id'].values)):
    
    case_id_index = list(mult_shap_dict['Samples']).index(slide)
    assert mult_shap_dict['Samples'][case_id_index] == slide, "Correct case ID not found in results."
    color_value_case = shap_spec_feat[case_id_index]

    # Get the RNA data for the specific case
    rna_data_case = rna_data[rna_data['Unnamed: 0'] == sample]

    pathway_name = hallmarks_pd.columns.tolist()[pathway]
    pathway_name = ' '.join(pathway_name[9:].split('_'))

    genes = hallmarks_pd[hallmarks_pd.columns[pathway]].values

    genes = rna_data_case.columns.intersection(genes)
    all_expression_data = rna_data_case[genes]
    # Calculate the mean expression of the pathway (for this sample)
    mean_expression = np.mean(all_expression_data)
    rna_vals.append(mean_expression)
    shap_vals.append(color_value_case)



In [ ]:
# Convert to dataframe
plot_data = pd.DataFrame({
    "Mean expression": rna_vals,
    "Color value": shap_vals,
    "Pathway": "R13"  # replace with actual name or list of pathway names
})

# Swarm plot
pathway_swarm_plot(plot_data)

# Correlation table
sp_r, sp_p = spearmanr(rna_vals, shap_vals)
pe_r, pe_p = pearsonr(rna_vals, shap_vals)

corr_df = pd.DataFrame([{
    "Spearman r": round(sp_r, 3),
    "Spearman p": sp_p,
    "Pearson r": round(pe_r, 3),
    "Pearson p": pe_p, 
}])

print(corr_df.to_string(index=False))

# Plot pathway distribution

In [ ]:
# Put your hyperparmeters here!!

case = 'TCGA-3C-AALK'

pathway_index = 48

In [ ]:
# Obtain train data
train_data = pd.read_csv(f'../data/data_files/tcga_{type}/splits/{fold}/train_filtered.csv')
train_cases = train_data['case_id'].values

In [ ]:
# Find case id
case_index = np.where(np.array(shap_dict['Samples']) == case)[0][0]
shap_values_case = shap_values_rna[case_index]

# Get data for the ridgeplot
data_ridge_plot = get_data_ridge(pathway_index, case, shap_values_case, hallmarks_pd, rna_data, train_cases)

# Plot ridge
plot_ridge_pathways(data_ridge_plot)